In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import json
from pathlib import Path
from collections import Counter

def explore_coco_json(json_path, max_items=3, max_depth=3):
    """
    探索并展示COCO JSON文件的层次结构
    
    Args:
        json_path: JSON文件路径
        max_items: 每个层级最多显示的项目数
        max_depth: 最大探索深度
    """
    try:
        print(f"正在加载文件: {json_path}")
        print("=" * 60)
        
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print("✅ 文件加载成功！")
        print(f"文件大小: {Path(json_path).stat().st_size / 1024 / 1024:.2f} MB")
        print("=" * 60)
        
        def explore_structure(obj, depth=0, key=""):
            if depth > max_depth:
                return
                
            indent = "  " * depth
            prefix = f"{indent}├─ " if key else ""
            
            if isinstance(obj, dict):
                print(f"{indent}{prefix}{key}: DICT (包含 {len(obj)} 个键)")
                
                if depth < max_depth:
                    # 只显示前几个键
                    displayed_keys = list(obj.keys())[:max_items]
                    for k in displayed_keys:
                        explore_structure(obj[k], depth + 1, k)
                    
                    if len(obj) > max_items:
                        print(f"{indent}  └─ ... 还有 {len(obj) - max_items} 个键未显示")
            
            elif isinstance(obj, list):
                print(f"{indent}{prefix}{key}: LIST (包含 {len(obj)} 个项目)")
                
                if depth < max_depth and len(obj) > 0:
                    # 只显示前几个项目
                    for i, item in enumerate(obj[:max_items]):
                        explore_structure(item, depth + 1, f"[{i}]")
                    
                    if len(obj) > max_items:
                        print(f"{indent}  └─ ... 还有 {len(obj) - max_items} 个项目未显示")
                    
                    # 如果是深层结构，显示第一个项目的详细信息
                    if depth == 1 and len(obj) > 0:
                        first_item = obj[0]
                        if isinstance(first_item, dict):
                            print(f"\n{indent}第一个项目的键示例:")
                            for k in list(first_item.keys())[:min(10, len(first_item))]:
                                v = first_item[k]
                                v_type = type(v).__name__
                                print(f"{indent}  {k}: {v_type} = {str(v)[:50]}{'...' if len(str(v)) > 50 else ''}")
            
            else:
                value_str = str(obj)
                if len(value_str) > 50:
                    value_str = value_str[:47] + "..."
                print(f"{indent}{prefix}{key}: {type(obj).__name__} = {value_str}")
        
        print("📊 JSON文件结构概览:")
        explore_structure(data, key="ROOT")
        print("=" * 60)
        
        # COCO特定的分析
        if isinstance(data, dict):
            print("🔍 COCO数据结构详细分析:")
            
            # 1. 基本信息
            if 'info' in data:
                print("\n1. 数据集信息 (info):")
                for k, v in data['info'].items():
                    print(f"   {k}: {v}")
            
            # 2. 图像列表
            if 'images' in data:
                images = data['images']
                print(f"\n2. 图像列表 (images):")
                print(f"   总图像数: {len(images)}")
                
                if images:
                    first_image = images[0]
                    print(f"   第一个图像示例:")
                    for k, v in first_image.items():
                        print(f"     {k}: {v}")
                    
                    # 统计关键字段
                    licenses = [img.get('license', 'unknown') for img in images[:100]]
                    license_counter = Counter(licenses)
                    print(f"\n   前100张图像的license分布: {dict(license_counter)}")
            
            # 3. 标注列表
            if 'annotations' in data:
                annotations = data['annotations']
                print(f"\n3. 标注列表 (annotations):")
                print(f"   总标注数: {len(annotations)}")
                
                if annotations:
                    first_ann = annotations[0]
                    print(f"   第一个标注示例:")
                    for k, v in first_ann.items():
                        if k == 'caption':
                            caption = str(v)
                            print(f"     {k}: {caption[:80]}{'...' if len(caption) > 80 else ''}")
                        else:
                            print(f"     {k}: {v}")
                    
                    # 标注统计
                    unique_images = len(set(ann['image_id'] for ann in annotations))
                    print(f"\n   标注覆盖的图像数: {unique_images}")
                    print(f"   平均每张图像的标注数: {len(annotations) / unique_images:.1f}")
            
            # 4. 图像-标注关系分析
            if 'images' in data and 'annotations' in data:
                print(f"\n4. 图像与标注关系:")
                print(f"   图像总数: {len(data['images'])}")
                print(f"   标注总数: {len(data['annotations'])}")
                
                # 计算每张图有多少个标注
                image_to_caption_count = {}
                for ann in data['annotations']:
                    img_id = ann['image_id']
                    image_to_caption_count[img_id] = image_to_caption_count.get(img_id, 0) + 1
                
                if image_to_caption_count:
                    avg_captions = sum(image_to_caption_count.values()) / len(image_to_caption_count)
                    print(f"   有标注的图像数: {len(image_to_caption_count)}")
                    print(f"   平均每张图标注数: {avg_captions:.1f}")
                    print(f"   最少标注的图像有: {min(image_to_caption_count.values())} 个标注")
                    print(f"   最多标注的图像有: {max(image_to_caption_count.values())} 个标注")
        
        print("=" * 60)
        print("🎯 关键发现:")
        print("1. 确认是否有 'annotations' 键，其中应包含 'caption' 字段")
        print("2. 检查 'images' 和 'annotations' 如何通过 'image_id' 关联")
        print("3. 验证每个图像都有多个标注描述")
        
        return data
        
    except FileNotFoundError:
        print(f"❌ 文件未找到: {json_path}")
        print("请检查文件路径是否正确。Kaggle中的典型路径是:")
        print("  /kaggle/input/coco-2014/captions_train2014.json")
        print("  /kaggle/input/coco2014/captions/annotations/captions_train2014.json")
        return None
    except json.JSONDecodeError as e:
        print(f"❌ JSON解析错误: {e}")
        return None
    except Exception as e:
        print(f"❌ 发生错误: {e}")
        return None


# 你的文件路径
json_path = "/kaggle/input/coco2014/captions/annotations/captions_train2014.json"

# 可选：如果上面的路径不存在，尝试这些常见的Kaggle路径
alternative_paths = [
    "/kaggle/input/coco-2014/captions_train2014.json",
    "/kaggle/input/coco-2014/annotations/captions_train2014.json",
    "/kaggle/input/coco2014/annotations/captions_train2014.json",
    "/kaggle/input/coco-2014/captions/annotations/captions_train2014.json"
]

# 尝试多个路径
paths_to_try = [json_path] + alternative_paths

data = None
for path in paths_to_try:
    print(f"\n尝试路径: {path}")
    data = explore_coco_json(path)
    if data is not None:
        break
    print("-" * 60)

if data is None:
    print("\n❌ 所有路径都未找到COCO标注文件。")
    print("\n💡 解决方案:")
    print("1. 在Kaggle中，点击右侧的 'Add Data'")
    print("2. 搜索 'COCO 2014' 或 'COCO Captions'")
    print("3. 添加数据集后，查看文件路径并更新代码中的路径")
    print("\n或者，使用以下命令查看当前目录结构:")
    print("""
import os
print("当前目录内容:")
for root, dirs, files in os.walk('/kaggle/input'):
    for dir in dirs:
        print(f"目录: {os.path.join(root, dir)}")
    for file in files:
        if 'caption' in file.lower() or 'annot' in file.lower():
            print(f"标注文件: {os.path.join(root, file)}")
    """)